In [1]:
import sqlite3

conn = sqlite3.connect("earthquake.db") 
cursor = conn.cursor()

sql_script = """
-- ============================================================
-- Team 5071 | Week 8: Query Optimization
-- File: after_optimization.sql
-- Description: Indexes added to the baseline database, followed
--              by EXPLAIN QUERY PLAN output showing what changed.
-- ============================================================

CREATE INDEX IF NOT EXISTS idx_seismic_event_id
    ON SEISMIC_EVENT(event_id);

CREATE INDEX IF NOT EXISTS idx_magnitude_event
    ON MAGNITUDE_MEASUREMENT(event_id);

CREATE INDEX IF NOT EXISTS idx_review_status_name
    ON REVIEW_STATUS(status_name);

CREATE INDEX IF NOT EXISTS idx_magnitude_value
    ON MAGNITUDE_MEASUREMENT(magnitude_value);

CREATE INDEX IF NOT EXISTS idx_event_timestamp
    ON SEISMIC_EVENT(event_timestamp);
"""

# Run index creation
cursor.executescript(sql_script)
conn.commit()

# ----------------------------
# QUERY 1
# ----------------------------
print("\nQUERY 1 EXPLAIN:")
cursor.execute("""
EXPLAIN QUERY PLAN
SELECT S.event_id, G.latitude, G.longitude, M.magnitude_value
FROM GEOGRAPHIC_LOCATION AS G
INNER JOIN SEISMIC_EVENT AS S ON G.location_id = S.location_id
INNER JOIN MAGNITUDE_MEASUREMENT AS M ON S.event_id = M.event_id
WHERE S.event_id = 'us7000rn4a';
""")
for row in cursor.fetchall():
    print(row)

# ----------------------------
# QUERY 2
# ----------------------------
print("\nQUERY 2 EXPLAIN:")
cursor.execute("""
EXPLAIN QUERY PLAN
SELECT se.event_id, se.event_timestamp, rs.status_name AS review_status,
       gl.place_description, mm.magnitude_value
FROM SEISMIC_EVENT se
JOIN REVIEW_STATUS rs ON se.review_status_id = rs.status_id
JOIN GEOGRAPHIC_LOCATION gl ON se.location_id = gl.location_id
JOIN MAGNITUDE_MEASUREMENT mm ON se.event_id = mm.event_id
WHERE rs.status_name <> 'reviewed'
ORDER BY se.event_timestamp DESC;
""")
for row in cursor.fetchall():
    print(row)

# ----------------------------
# QUERY 3
# ----------------------------
print("\nQUERY 3 EXPLAIN:")
cursor.execute("""
EXPLAIN QUERY PLAN
SELECT 
    se.event_id,
    se.event_timestamp,
    gl.place_description,
    mm.magnitude_value,
    (SELECT ROUND(AVG(magnitude_value), 2) 
     FROM MAGNITUDE_MEASUREMENT) AS average_magnitude
FROM SEISMIC_EVENT se
JOIN GEOGRAPHIC_LOCATION gl ON se.location_id = gl.location_id
JOIN MAGNITUDE_MEASUREMENT mm ON se.event_id = mm.event_id
WHERE mm.magnitude_value > (
    SELECT AVG(magnitude_value) 
    FROM MAGNITUDE_MEASUREMENT
)
ORDER BY mm.magnitude_value DESC;
""")
for row in cursor.fetchall():
    print(row)

conn.close()


QUERY 1 EXPLAIN:
(6, 0, 0, 'SEARCH S USING INDEX sqlite_autoindex_SEISMIC_EVENT_1 (event_id=?)')
(11, 0, 0, 'SEARCH G USING INTEGER PRIMARY KEY (rowid=?)')
(14, 0, 0, 'SEARCH M USING INDEX idx_magnitude_event (event_id=?)')

QUERY 2 EXPLAIN:
(8, 0, 0, 'SCAN se USING INDEX idx_event_timestamp')
(11, 0, 0, 'SEARCH rs USING INTEGER PRIMARY KEY (rowid=?)')
(16, 0, 0, 'SEARCH gl USING INTEGER PRIMARY KEY (rowid=?)')
(19, 0, 0, 'SEARCH mm USING INDEX idx_magnitude_event (event_id=?)')

QUERY 3 EXPLAIN:
(7, 0, 0, 'SEARCH mm USING INDEX idx_magnitude_value (magnitude_value>?)')
(11, 0, 0, 'SCALAR SUBQUERY 2')
(16, 11, 0, 'SCAN MAGNITUDE_MEASUREMENT USING COVERING INDEX idx_magnitude_value')
(31, 0, 0, 'SEARCH se USING INDEX sqlite_autoindex_SEISMIC_EVENT_1 (event_id=?)')
(36, 0, 0, 'SEARCH gl USING INTEGER PRIMARY KEY (rowid=?)')
(46, 0, 0, 'SCALAR SUBQUERY 1')
(51, 46, 0, 'SCAN MAGNITUDE_MEASUREMENT USING COVERING INDEX idx_magnitude_value')


In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("cleaned_data.csv")

#### Question 1: What is the latitude, longitude, and magnitude of the earthquake with event_id = us7000rn4a? 

Baseline:

In the baseline version, we filter the DataFrame using a boolean condition on the event_id column. This approach scans the entire event_id column and compares every value to the target ID. While this works correctly, it becomes inefficient as the dataset grows because it performs a full column scan.


In [4]:
result = df[df["event_id"] == "us7000rn4a"][
    ["event_id", "latitude", "longitude", "magnitude_value"]
]

print(result)

         event_id  latitude  longitude  magnitude_value
37032  us7000rn4a    7.2863   127.0595              5.5


Optimized (using Index)

To optimize this lookup, we set event_id as the DataFrame index. Setting an index allows pandas to perform fast label-based lookups using .loc[] rather than scanning the entire column. The optimized version eliminates the need to scan the full event_id column. Instead, pandas performs a direct index lookup, which is significantly faster on large datasets.

In [5]:
df_indexed = df.set_index("event_id")

In [6]:
result = df_indexed.loc["us7000rn4a"][
    ["latitude", "longitude", "magnitude_value"]
]

print(result)

latitude             7.2863
longitude          127.0595
magnitude_value         5.5
Name: us7000rn4a, dtype: object


#### Question 2: What earthquakes had a magnitude greater than the average?

Baseline:

In the baseline version, we first compute the average magnitude across all earthquakes. Then, we filter the DataFrame using a boolean mask to select rows where the magnitude is greater than that average. Although this method is vectorized, it still evaluates the condition across the full column and creates an intermediate boolean mask.

In [7]:
avg_mag = df["magnitude_value"].mean()

above_avg = df[df["magnitude_value"] > avg_mag][
    ["event_id", "event_timestamp", "place_description", "magnitude_value"]
]

above_avg.head()

,event_id,event_timestamp,place_description,magnitude_value
0,ak000126digo,2000-01-23 08:42:28.405000+00:00,"190 km E of Chiniak, Alaska",5.5
1,ak0001kedehd,2000-02-03 10:24:57.773000+00:00,northern Alaska,5.6
9,ak0006fvclqz,2000-05-19 20:34:28.111000+00:00,"63 km WSW of Nanwalek, Alaska",5.8
11,ak0008v7ic1n,2000-07-11 01:32:28.758000+00:00,"24 km SSW of Larsen Bay, Alaska",6.5
12,ak0008v7jnot,2000-07-11 01:38:46.211000+00:00,"23 km SSE of Karluk, Alaska",5.5


Optimized:

To improve performance, we use the .query() method. The .query() method evaluates expressions more efficiently and avoids creating explicit intermediate boolean masks. It also makes the filtering logic easier to read and maintain.

In [8]:
avg_mag = df["magnitude_value"].mean()

above_avg = (
    df.query("magnitude_value > @avg_mag")
      [["event_id", "event_timestamp", "place_description", "magnitude_value"]]
)

above_avg.head()

,event_id,event_timestamp,place_description,magnitude_value
0,ak000126digo,2000-01-23 08:42:28.405000+00:00,"190 km E of Chiniak, Alaska",5.5
1,ak0001kedehd,2000-02-03 10:24:57.773000+00:00,northern Alaska,5.6
9,ak0006fvclqz,2000-05-19 20:34:28.111000+00:00,"63 km WSW of Nanwalek, Alaska",5.8
11,ak0008v7ic1n,2000-07-11 01:32:28.758000+00:00,"24 km SSW of Larsen Bay, Alaska",6.5
12,ak0008v7jnot,2000-07-11 01:38:46.211000+00:00,"23 km SSE of Karluk, Alaska",5.5


#### Question 3: How can we optimize memory usage for repeated text values?

Baseline:

In the baseline version, the place_description column is stored as string data type. Object columns consume significantly more memory because each value is stored as a full Python string. Since many earthquake records share repeated location names, this leads to unnecessary memory usage.

In [9]:
memory_before = df["place_description"].memory_usage(deep=True)

memory_before

9451391

Optimized:

To improve efficiency, we convert place_description to a categorical data type. Categorical columns store unique values once and reference them internally using integer codes. This reduces memory usage and improves performance when filtering or grouping by that column.

In [10]:
df["place_description"] = df["place_description"].astype("category")

memory_after = df["place_description"].memory_usage(deep=True)

memory_after 

8637898

In [11]:
percent_reduction = ((memory_before - memory_after) / memory_before) * 100

print(f"Memory reduced by {percent_reduction:.2f}%")

Memory reduced by 8.61%
